# Dia 5 — Orquestrador com Skills e Sub-agente

O problema que resolvemos:
> O usuário pede genericamente *"envia um alerta de falha"*.  
> O sistema precisa saber **como** escrever esse alerta — tom, formato, campos obrigatórios.  
> Esse conhecimento fica em arquivos `.md` chamados **skills**.

### Arquitetura

```
Setup: lê cabeçalhos das skills → injeta no system_prompt do SUB-AGENTE

Usuário
   ↓ pedido genérico
Orquestrador  (sabe apenas que pode chamar consultar_skill)
   ↓ consultar_skill("tipo de alerta")
   ↓   └─ Sub-agente de skills  (sabe quais skills existem)
   ↓         └─ read_skill(header_only=False, filename=...) → instruções completas
   ↓ listar_destinatarios() → obtém lista de usuários
   ↓ send_email() × N → envia para cada destinatário
```

| Parte | Tema |
|---|---|
| Setup | Conexões, credenciais, skills em disco |
| A | Tool `read_skill` |
| B | Injetar headers no system prompt do sub-agente |
| C | Sub-agente de skills |
| D | Orquestrador |
| E | Testes end-to-end |

---

## Setup

In [ ]:
!pip install -q langchain-anthropic langchain langgraph

In [ ]:
PROXY_URL      = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN    = "xpto_aluno-01"

EMAIL_URL      = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN    = "aluno-01"
EMAIL_PASSWORD = "1234"

# Pasta onde as skills serão salvas
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SKILLS_FOLDER = "/content/drive/MyDrive/curso_agentes/skills"
    print(f"Google Drive montado. Skills em: {SKILLS_FOLDER}")
except ImportError:
    SKILLS_FOLDER = "skills"
    print(f"Executando localmente. Skills em: {SKILLS_FOLDER}")

print("Credenciais carregadas.")

In [ ]:
import os

os.makedirs(SKILLS_FOLDER, exist_ok=True)

SKILL_FALHA = """# skill: alerta_falha
# descricao: Instrui como redigir um e-mail de alerta de falha crítica no sistema.
# palavras-chave: falha, erro, crítico, down, indisponível

---

## Instruções para o agente orquestrador

Você deve redigir um e-mail de **alerta de falha crítica** seguindo rigorosamente o formato abaixo.

### Formato obrigatório do corpo do e-mail

🚨 ALERTA DE FALHA CRÍTICA

**Sistema afetado:** [nome do sistema ou componente]
**Severidade:** CRÍTICA
**Horário da ocorrência:** [horário informado ou "não informado"]

**O que aconteceu:**
[descrição breve da falha em 1-2 frases]

**Impacto:**
[descreva quem ou o que é afetado]

**Ação necessária:**
[o que o time deve fazer agora]

— Sistema de Monitoramento Automático

### Regras de formatação
- Sempre iniciar com o emoji 🚨 e o título em CAIXA ALTA
- O campo Severidade deve ser sempre CRÍTICA
- Assunto do e-mail deve seguir o padrão: [FALHA] <nome do sistema>
- Destinatários: todos os usuários da lista fornecida pelo orquestrador
"""

SKILL_PRODUCAO = """# skill: alerta_producao
# descricao: Instrui como redigir um e-mail comemorativo de meta de produção atingida.
# palavras-chave: produção, meta, atingida, sucesso, meta batida, resultado

---

## Instruções para o agente orquestrador

Você deve redigir um e-mail de **celebração de meta de produção atingida** seguindo o formato abaixo.

### Formato obrigatório do corpo do e-mail

✅ META DE PRODUÇÃO ATINGIDA

**Linha / Área:** [nome da linha ou área]
**Meta estabelecida:** [valor da meta]
**Resultado alcançado:** [valor atingido]
**Período:** [período informado ou "turno atual"]

Parabéns ao time! 🎉
[mensagem motivacional curta de 1-2 frases]

**Próximos passos:**
[o que acontece agora]

— Sistema de Monitoramento de Produção

### Regras de formatação
- Sempre iniciar com o emoji ✅ e o título em CAIXA ALTA
- O campo Resultado alcançado deve aparecer em destaque
- Assunto do e-mail deve seguir o padrão: [PRODUÇÃO] Meta atingida — <área>
- Destinatários: todos os usuários da lista fornecida pelo orquestrador
"""

with open(f"{SKILLS_FOLDER}/alerta_falha.md", "w", encoding="utf-8") as f:
    f.write(SKILL_FALHA)

with open(f"{SKILLS_FOLDER}/alerta_producao.md", "w", encoding="utf-8") as f:
    f.write(SKILL_PRODUCAO)

print(f"Skills criadas em '{SKILLS_FOLDER}':")
for a in sorted(os.listdir(SKILLS_FOLDER)):
    print(f"  - {a}")

In [ ]:
import requests
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=1024,
)

resp = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print("LLM:", resp.content)

login = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN, "password": EMAIL_PASSWORD,
})
assert login.status_code == 200
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Email server: OK →", login.json())

In [ ]:
from langchain_core.tools import tool
from typing import Optional

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: E-mails em cópia, separados por vírgula (opcional)
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc
    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

print("Tool send_email pronta.")

---
## Parte A — Tool `read_skill`

A tool do orquestrador para ler as instruções completas de uma skill.

- `header_only=True` → scan de cabeçalhos de todos os arquivos (usado no setup para montar o system prompt)
- `header_only=False` + `filename` → lê o conteúdo completo de um arquivo específico (usado pelo orquestrador em tempo de execução)

In [ ]:
HEADER_LINES = 4

@tool
def read_skill(header_only: bool, filename: Optional[str] = None) -> str:
    """Lê arquivos de skill (.md) da pasta de skills.

    Use header_only=False para ler as instruções completas de uma skill específica.
    O orquestrador já sabe quais arquivos existem via system_prompt.

    Args:
        header_only: True para escanear cabeçalhos, False para ler completo.
        filename: Nome do arquivo .md (necessário quando header_only=False).
    """
    if header_only:
        resultado = []
        try:
            arquivos = sorted(f for f in os.listdir(SKILLS_FOLDER) if f.endswith(".md"))
        except FileNotFoundError:
            return f"Pasta de skills não encontrada: '{SKILLS_FOLDER}'"
        if not arquivos:
            return "Nenhuma skill encontrada na pasta."
        for arq in arquivos:
            caminho = os.path.join(SKILLS_FOLDER, arq)
            with open(caminho, "r", encoding="utf-8") as f:
                linhas = [f.readline() for _ in range(HEADER_LINES)]
            resultado.append(f"=== {arq} ===")
            resultado.append("".join(linhas).strip())
            resultado.append("")
        return "\n".join(resultado)
    else:
        if not filename:
            return "Erro: filename é obrigatório quando header_only=False."
        caminho = os.path.join(SKILLS_FOLDER, filename)
        if not os.path.exists(caminho):
            return f"Arquivo '{filename}' não encontrado em '{SKILLS_FOLDER}'."
        with open(caminho, "r", encoding="utf-8") as f:
            return f.read()

# Teste direto
print(read_skill.invoke({"header_only": True}))

---
## Parte B — Injetar headers no system prompt do sub-agente

Lemos os cabeçalhos **uma vez no setup** e injetamos no `system_prompt` do **sub-agente**.

O sub-agente precisa saber quais arquivos existem para escolher o correto.  
O orquestrador **não precisa saber disso** — ele só sabe que pode chamar `consultar_skill`.

In [ ]:
def montar_contexto_skills() -> str:
    """Lê os cabeçalhos de todas as skills e monta o bloco para o system prompt."""
    try:
        arquivos = sorted(f for f in os.listdir(SKILLS_FOLDER) if f.endswith(".md"))
    except FileNotFoundError:
        return ""
    if not arquivos:
        return ""

    linhas = ["Você tem acesso às seguintes skills:\n"]
    for arq in arquivos:
        caminho = os.path.join(SKILLS_FOLDER, arq)
        with open(caminho, "r", encoding="utf-8") as f:
            header = "".join(f.readline() for _ in range(HEADER_LINES))
        linhas.append(f"- Arquivo: {arq}\n{header.strip()}\n")

    linhas.append(
        "Quando precisar das instruções completas de uma skill, "
        "use read_skill(header_only=False, filename='nome_do_arquivo.md')."
    )
    return "\n".join(linhas)


SKILLS_CONTEXT = montar_contexto_skills()
print("=== Contexto injetado no system prompt ===")
print(SKILLS_CONTEXT)

---
## Parte C — Sub-agente de skills

O sub-agente tem acesso **apenas** à `read_skill` — ele não envia e-mails.  
O orquestrador o chama via tool `consultar_skill`, sem saber como a leitura funciona internamente.

> **Por que usar um sub-agente aqui?**  
> A leitura de uma skill envolve decisão (qual arquivo?) e execução (ler o conteúdo).  
> Delegar isso a um agente especializado mantém o orquestrador focado em coordenar e agir.

In [ ]:
sub_agente_skills = create_agent(
    model=llm,
    tools=[read_skill],
    system_prompt=(
        "Você é um sub-agente especializado em leitura de skills.\n"
        "Dado um tipo de alerta, encontre e retorne as instruções completas da skill correta.\n"
        "Use read_skill(header_only=False, filename=...) para ler o arquivo.\n"
        f"{SKILLS_CONTEXT}"
    ),
)

print("Sub-agente de skills criado.")

In [ ]:
@tool
def consultar_skill(tipo_de_alerta: str) -> str:
    """Consulta o sub-agente de skills para obter as instruções completas de formatação.

    Use esta tool para saber COMO escrever um e-mail de alerta.

    Args:
        tipo_de_alerta: Descrição do tipo de alerta (ex: "alerta de falha crítica")
    """
    resultado = sub_agente_skills.invoke(
        {"messages": [{"role": "user", "content": tipo_de_alerta}]},
        config={"recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("Tool consultar_skill pronta.")

---
## Parte D — Orquestrador

O orquestrador tem três tools: `consultar_skill`, `listar_destinatarios` e `send_email`.  
Ele delega a leitura de instruções ao sub-agente via `consultar_skill` e se mantém focado em coordenar e agir.

In [ ]:
TODOS_USUARIOS = [
    "aluno01@curso.ia",
    "aluno02@curso.ia",
    "aluno03@curso.ia",
    "aluno04@curso.ia",
    "aluno05@curso.ia",
]

@tool
def listar_destinatarios() -> str:
    """Retorna a lista de todos os usuários que devem receber alertas do sistema.

    Use esta tool quando o usuário pedir para enviar para 'todos' ou 'todos os usuários'.
    """
    return "Destinatários:\n" + "\n".join(f"- {u}" for u in TODOS_USUARIOS)

print("Tool listar_destinatarios pronta.")

In [ ]:
from langchain.agents import create_agent

orquestrador = create_agent(
    model=llm,
    tools=[consultar_skill, listar_destinatarios, send_email],
    system_prompt=(
        "Você é o agente orquestrador de alertas.\n"
        "Quando o usuário pedir para enviar um alerta, siga sempre este processo:\n\n"
        "1. Use consultar_skill(tipo_de_alerta=...) para obter as instruções completas de formatação.\n"
        "2. Use listar_destinatarios para obter a lista de usuários.\n"
        "3. Redija o e-mail seguindo RIGOROSAMENTE as instruções retornadas.\n"
        "4. Envie o e-mail para CADA destinatário usando send_email.\n\n"
        "Você não precisa saber quais skills existem — o sub-agente cuida disso."
    ),
)

print("Orquestrador criado.")

---
## Parte E — Testes end-to-end

In [ ]:
def invocar_orquestrador(prompt: str) -> str:
    resultado = orquestrador.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 20},
    )
    return resultado["messages"][-1].content

print("Helper pronto.")

In [ ]:
# Teste 1: alerta de falha
print("=== ALERTA DE FALHA ===")
resposta = invocar_orquestrador(
    "Envia um alerta de falha. O sistema de pagamentos está fora do ar desde as 14h."
)
print(resposta)

In [ ]:
# Teste 2: alerta de produção atingida
print("=== META DE PRODUÇÃO ATINGIDA ===")
resposta = invocar_orquestrador(
    "Envia o alerta de produção. A linha 3 bateu a meta de 500 unidades neste turno."
)
print(resposta)

In [ ]:
# Verificar e-mails recebidos
r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
inbox = r.json()
print(f"{inbox['count']} mensagem(ns) na caixa de entrada:\n")
for msg in inbox["messages"]:
    print(f"  [{msg['id']}] De: {msg['from']:<25} | {msg['subject']}")

---
## Resumo do Dia 5 e do Curso

| Conceito | O que aprendemos |
|---|---|
| Skill (`.md`) | Arquivo de instrução que ensina o agente a formatar uma mensagem |
| Headers no system prompt | Cabeçalhos injetados no setup — o agente já sabe o que existe |
| `read_skill` | Tool para ler instruções completas em tempo de execução |
| Sub-agente de skills | Agente especializado que isola a lógica de leitura de skills |
| `consultar_skill` | Tool que expõe o sub-agente ao orquestrador como uma única chamada |
| Orquestrador | Agente que delega para o sub-agente e coordena o envio |

---

## Arco completo do curso

| Dia | Conceito central |
|---|---|
| 1 | Conexão ao LLM + primeira tool + primeiro agente |
| 2 | Múltiplas tools + agente multi-tool + memória persistente |
| 3 | RAG com FAISS — o agente consulta conhecimento externo |
| 4 | LangGraph — controle explícito de fluxo + human-in-the-loop |
| 5 | Skills em `.md` + headers no system prompt + sub-agente + orquestrador |